In [26]:
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [28]:
loader = PyPDFLoader("../data/admit.pdf")
docs = loader.load()

In [29]:
len(docs)

1

In [30]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)


In [31]:
len(splitted_docs)

4

In [32]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [33]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context


In [34]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-latest",
    temperature=0.7
)

In [35]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [36]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [37]:
query = "What is the name of candidate and whats it roll number ?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 18.353092383s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMod

In [ ]:
result = response["messages"][-1].content

NameError: name 'response' is not defined

In [ ]:
print(result)

NameError: name 'result' is not defined